# 1 · Spotiphy deconvolution and decomposition

HyperSpot works on **inferred single cells**. Spotiphy ([Yang et al.](https://github.com/jyyulab/Spotiphy))
estimates cell-type proportions per Visium spot, counts nuclei on the H&E image, and decomposes each
spot's expression into per-cell profiles with generated cell locations.

Kernel: `Spotiphy` environment (GPU recommended). Outputs used later:
`ST_decomposition.h5ad`, `proportion.npy` and the `segmentation/` folder.
Based on the [Spotiphy Visium tutorial](https://github.com/jyyulab/Spotiphy/blob/main/tutorials/Spotiphy_tutorial_visium.ipynb).

In [ ]:
import pickle
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import spotiphy
import torch

from pathlib import Path

# All tutorial outputs live next to these notebooks; run the notebooks from tutorial/.
DATA_DIR      = Path("../demo/data_repo/Human Kidney Cancer")     # raw inputs (see README)
PATIENT_ID    = "PD47171"
PROJECT       = "Human Kidney Cancer Core"
SPOTIPHY_DIR  = Path("spotiphy_project_files") / PROJECT          # Spotiphy inputs/outputs
HYPERSPOT_DIR = Path("hyperspot_project_files") / PROJECT         # HyperSpot project (the CLI resolves
                                                                  # --project relative to ./hyperspot_project_files)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## Load the inputs from notebook 0

In [ ]:
adata_sc = sc.read_h5ad(SPOTIPHY_DIR / "Human Kidney Cancer scRNA Ref.h5ad")
adata_st = sc.read_h5ad(SPOTIPHY_DIR / f"{PROJECT}.h5ad")
img = cv2.imread(str(SPOTIPHY_DIR / f"{PROJECT}.tif"))      # BGR

key_type = "broad_type"
type_list = sorted(adata_sc.obs[key_type].unique().astype(str))
adata_st.var_names_make_unique()
adata_st.obsm["spatial"] = adata_st.obsm["spatial"].astype(np.int32)
print(f"{len(type_list)} cell types: {type_list}")

## Marker genes and single-cell reference

`initialization` normalises both datasets and keeps shared genes. Markers are selected by pairwise tests
(up to 50 per cell type).

In [ ]:
adata_sc, adata_st = spotiphy.initialization(adata_sc, adata_st, verbose=1)

marker_gene_dict = spotiphy.sc_reference.marker_selection(
    adata_sc, key_type=key_type, return_dict=True,
    n_select=50, threshold_p=0.1, threshold_fold=1.5, q=0.15)
marker_gene = [g for t in type_list for g in marker_gene_dict[t]]
pd.DataFrame({"gene": marker_gene,
              "label": [t for t in type_list for _ in marker_gene_dict[t]]}).to_csv(SPOTIPHY_DIR / "marker_gene.csv")

adata_sc_marker = adata_sc[:, marker_gene]
adata_st_marker = adata_st[:, marker_gene]
sc_ref = spotiphy.construct_sc_ref(adata_sc_marker, key_type=key_type)

## Cell-type proportions per spot

In [ ]:
cell_proportion = spotiphy.deconvolution.estimation_proportion(
    np.array(adata_st_marker.X), adata_sc_marker, sc_ref, type_list, key_type,
    n_epoch=8000, plot=True, batch_prior=1, device=device)
adata_st.obs[type_list] = cell_proportion
np.save(SPOTIPHY_DIR / "proportion.npy", cell_proportion)

In [ ]:
vmax = np.maximum(np.quantile(adata_st.obs[type_list].values, 0.98, axis=0), 0.05)
sc.pl.spatial(adata_st, color=type_list, cmap="viridis", vmin=0, vmax=list(vmax),
              img_key="hires", size=1.3, ncols=6, alpha_img=0.4)

## Nucleus segmentation (cells per spot)

StarDist segments nuclei on the H&E image. Increase `n_tiles` if you run out of memory.

In [ ]:
segmentation_dir = SPOTIPHY_DIR / "segmentation"
segmentation_dir.mkdir(exist_ok=True)

segmentation = spotiphy.segmentation.Segmentation(
    img[:, :, [2, 1, 0]], adata_st.obsm["spatial"],
    n_tiles=(32, 32, 1), out_dir=str(segmentation_dir) + "/", enhancement=True)   # Spotiphy concatenates paths
segmentation.segment_nucleus(save=True)
n_cell_df = pd.DataFrame(segmentation.n_cell_df)   # also saved by Spotiphy as segmentation/n_cell_df.csv

## Decomposition to single-cell resolution

Each spot's full expression profile is split across its estimated cells. The decomposed data is
HyperSpot's input.

In [ ]:
adata_sc_orig = sc.read_h5ad(SPOTIPHY_DIR / "Human Kidney Cancer scRNA Ref.h5ad")
adata_st_orig = sc.read_h5ad(SPOTIPHY_DIR / f"{PROJECT}.h5ad")
adata_st_orig.var_names_make_unique()

adata_decomp = spotiphy.deconvolution.decomposition(
    adata_st_orig, adata_sc_orig, key_type, cell_proportion,
    save=True, out_dir=str(SPOTIPHY_DIR) + "/", filename="ST_decomposition.h5ad",
    verbose=1, spot_location=adata_st_orig.obsm["spatial"],
    filtering_gene=True, n_cell=n_cell_df["cell_count"].values)
adata_decomp.var["SYMBOL"] = adata_sc_orig.var["name"]           # keep gene symbols for plots
adata_decomp.write(SPOTIPHY_DIR / "ST_decomposition.h5ad")
adata_decomp

## Cell boundaries and cell-type assignment

These steps place each inferred cell at a nucleus position. HyperSpot reads these generated locations
(`common.add_generated_cell_locations` in notebook 2) from the saved `segmentation.pkl`.

In [ ]:
boundary_dict = spotiphy.segmentation.cell_boundary(
    segmentation.nucleus_df[["x", "y"]].values, img_size=img.shape[:2],
    max_dist=25, max_area=580, verbose=0, delta=2,
    search_direction=[[1, 0], [0, 1], [-1, 0], [0, -1], [2, 0], [0, 2], [-2, 0], [0, -2],
                      [1, 1], [-1, 1], [-1, -1], [1, -1]])
with open(segmentation_dir / "boundary_dict.pkl", "wb") as f:
    pickle.dump(boundary_dict, f)

cell_number = spotiphy.deconvolution.proportion_to_count(
    cell_proportion, n_cell_df["cell_count"].values, multiple_spots=True)
segmentation.nucleus_df = spotiphy.deconvolution.assign_type_spot(
    segmentation.nucleus_df, segmentation.n_cell_df, cell_number, type_list)
segmentation.nucleus_df, _ = spotiphy.deconvolution.assign_type_out(
    segmentation.nucleus_df, cell_proportion, segmentation.spot_center, type_list, band_width=150)

with open(segmentation_dir / "segmentation.pkl", "wb") as f:
    pickle.dump(segmentation, f)

In [ ]:
plot_visium = spotiphy.plot.Plot_Visium(segmentation=segmentation, boundary_dict=boundary_dict, type_list=type_list)
plot_visium.plot_legend()
plot_visium.plot(background=False, spot_color=(200, 200, 200), shape="cell", cell="in", boundary="both", dpi=200)